# Un troisième témoin non génératif : le banc d'essai

Ce notebook établit, sur les **120 exemples** de [`corpus/demandes-rgpd.fr.jsonl`](../corpus/demandes-rgpd.fr.jsonl),
si un moteur de détection **fondé sur l'apprentissage automatique mais non génératif** apporte —
en tant que **témoin** aux côtés du lexique — une valeur d'alarme que le montage actuel à deux
moteurs n'a pas.

- Carte : [Un troisième moyen de détection, non génératif, pour affiner le diagnostic](https://github.com/AmauryTISSOT/microservice_rgpd/issues/42)
- Ticket : [Écrire le notebook d'exploration du troisième témoin](https://github.com/AmauryTISSOT/microservice_rgpd/issues/50)
- Lecture des chiffres et verdict : [Lire les chiffres du notebook et prononcer le verdict](https://github.com/AmauryTISSOT/microservice_rgpd/issues/52)

## Ce que ce notebook n'est pas

**Ce n'est pas une évaluation de classifieur.** Le candidat est jugé comme **témoin** : ce qui
compte n'est pas son exactitude mais sa capacité à **contredire un verdict faux sans contredire
un verdict juste**. Un tableau d'exactitude par classe est du contexte utile ; ce n'est pas la
réponse à la question posée.

**Ce n'est pas une redéfinition du `ReviewSignal`.** La règle d'alarme à trois avis est ici
**simulée**, pas promulguée. Le § 6.2 de la spec reste strictement binaire, et la carte a placé
hors périmètre le chantier de domaine — spec, contrat public, consommateurs.

**Ce n'est pas la décision.** Le notebook confronte les chiffres au critère fixé d'avance et
imprime ce que ce critère commande. Le verdict est prononcé ailleurs.

## Le critère, fixé avant tout chiffre

Fixé par [Fixer le critère de pertinence du troisième témoin](https://github.com/AmauryTISSOT/microservice_rgpd/issues/48),
**avant** l'écriture de ce notebook — c'est ce qui le rend opposable.

La règle d'alarme simulée est le **consensus**, pas l'union :

```
                    ┌─── lexique (L)
  qwen3:8b (V) ──≠?─┤        alarme  =  (V ≠ L)  ET  (V ≠ N)
                    └─── nouveau témoin (N)
```

> **OUI** si les trois conditions sont réunies :
>
> 1. La **borne haute** de l'intervalle de Wilson à 95 % du taux de fausse alarme, sur les **113
>    verdicts corrects** de `qwen3:8b`, en prédictions **hors-pli**, est **strictement inférieure
>    à 21,2 %** — soit **au plus 15 fausses alarmes**.
> 2. Les **5** erreurs `acc-10`, `hop-15`, `hop-22`, `lim-08`, `por-08` **restent toutes
>    signalées**. Condition binaire.
> 3. Les deux conditions tiennent au **pire des R = 5 germes**, pas en médiane.
>
> **NON** sinon — un intervalle qui chevauche 21,2 % est un non, pas une zone grise.
>
> **NON CONCLUANT** seulement si une condition de validité a échoué, et il faut dire laquelle.
> Liste **close** : bug d'accents confirmé · validation croisée impraticable (plis dégénérés,
> paires minimales inséparables, réglage qui ne converge pas) · prédictions constantes ou
> dégénérées.

## Les trois montages

Tranchés par [Trancher la famille d'approche et le modèle mis à l'épreuve](https://github.com/AmauryTISSOT/microservice_rgpd/issues/47).
Une **tête unique** aux trois — si la tête changeait, on ne mesurerait plus la représentation
mais un mélange.

| | Montage | Représentation | Rôle |
| --- | --- | --- | --- |
| 1 | `TfidfVectorizer` + tête | creuse, lexicale | **ligne de base** — le lexique automatisé |
| 2 | `intfloat/multilingual-e5-small` **gelé** + tête | dense, 384 d | **candidat principal** |
| 3 | **SetFit** sur le même encodeur + tête | dense, réentraînée par contraste | **échelon supérieur** |

Ce qui les départage n'est pas l'exactitude mais l'**indépendance** : le lexique est un moteur de
surface lexicale, TF-IDF est le même matériau, le plongement dense est la seule représentation
d'une autre nature.

## L'environnement d'exécution

Imprimé et versionné. Sans cette cellule, un écart de chiffres entre deux machines serait
indécidable ; avec elle, il est diagnosticable.

La reproductibilité visée est à deux niveaux, et c'est une décision, pas un renoncement
([Emplacement, outillage et reproductibilité du notebook](https://github.com/AmauryTISSOT/microservice_rgpd/issues/49)) :
**même machine + même `uv.lock` ⇒ les mêmes chiffres** ; **machine différente ⇒ le même verdict**
seulement. L'identité bit à bit sur CPU avec `torch` n'est pas atteignable — nombre de fils BLAS,
version de bibliothèque, jeu d'instructions — et la prétendre serait une promesse creuse.
L'ADR-0001 n'exige le déterminisme que du **service**, pas du banc d'essai qui le choisit.

Le corps contrastif du montage 3 est entraîné **sur la carte graphique quand il y en a une** ; tout
le reste, y compris la mesure de latence, tourne sur CPU. La raison est écrite dans la cellule de
coût : la latence rapportée est celle du **service**, et le service n'a pas de carte à lui. Le
niveau de reproductibilité s'en trouve précisé et non abaissé — l'identité des chiffres suppose
désormais la même machine, le même `uv.lock` **et le même appareil d'entraînement**. Un notebook
exécuté sans carte reste valide : il rend le même verdict, plus lentement.

In [1]:
import json, math, os, platform, random, sys, time, warnings
from collections import Counter
from pathlib import Path

# Bruit d'import et d'entraînement, sans rapport avec ce qui est mesuré. Filtré ici pour que les
# sorties versionnées restent lisibles — jamais pour taire un avertissement du banc d'essai.
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", message=".*pin_memory.*")

import numpy as np
import psutil
import sklearn
import torch
import transformers
import sentence_transformers
import setfit

print(f"python                {platform.python_version()}  ({platform.machine()})")
print(f"plateforme            {platform.platform()}")
print(f"processeur            {platform.processor()}")
print(f"cœurs logiques        {os.cpu_count()}   physiques {psutil.cpu_count(logical=False)}")
print(f"torch                 {torch.__version__}   fils {torch.get_num_threads()}")
print(f"scikit-learn          {sklearn.__version__}")
print(f"transformers          {transformers.__version__}")
print(f"sentence-transformers {sentence_transformers.__version__}")
transformers.logging.set_verbosity_error()

print(f"setfit                {setfit.__version__}")
# L'entraînement contrastif du montage 3 va sur la carte quand il y en a une. Ce n'est pas un
# réglage de confort : 25 entraînements saturant les cœurs physiques pendant près de six heures
# rendaient la machine inutilisable. Tout le reste — et notamment la latence, qui est le coût du
# *service* — reste mesuré sur CPU, parce que c'est là que le sidecar tournerait.
APPAREIL_ENTRAINEMENT = "cuda" if torch.cuda.is_available() else "cpu"

if APPAREIL_ENTRAINEMENT == "cuda":
    carte = torch.cuda.get_device_properties(0)
    vram_libre, vram_totale = torch.cuda.mem_get_info()
    print(f"CUDA                  {torch.version.cuda}   {carte.name} "
          f"(capacité {carte.major}.{carte.minor})")
    print(f"VRAM                  {vram_totale / 1024 ** 3:.1f} Gio dont "
          f"{vram_libre / 1024 ** 3:.1f} Gio libres")
    # Le corps contrastif a demandé jusqu'à ~4,2 Gio à la mesure. Une carte déjà occupée par
    # `qwen3:8b` (ADR-0001) ne laisserait pas de quoi : mieux vaut le savoir ici qu'au bout de la
    # vingtième minute.
    if vram_libre < 5 * 1024 ** 3:
        print("  ATTENTION : moins de 5 Gio libres — un serveur de modèles occupe-t-il la carte ?")
    # cuDNN chronomètre ses algorithmes pour choisir le plus rapide si on le laisse faire : deux
    # exécutions de la même cellule n'emprunteraient pas le même chemin. Le corps contrastif n'a
    # aucune convolution, la contrainte ne coûte donc rien ici.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
else:
    print("CUDA                  indisponible — le montage 3 retombe sur le CPU, et il faut "
          "alors compter plusieurs heures")

print(f"corps contrastif sur  {APPAREIL_ENTRAINEMENT}    |    latence mesurée sur  cpu")

python                3.13.5  (AMD64)
plateforme            Windows-11-10.0.26200-SP0
processeur            AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD
cœurs logiques        16   physiques 8
torch                 2.13.0+cu126   fils 8
scikit-learn          1.9.0
transformers          4.57.6
sentence-transformers 5.6.1
setfit                1.1.3


CUDA                  12.6   NVIDIA GeForce RTX 3070 Laptop GPU (capacité 8.6)
VRAM                  8.0 Gio dont 7.0 Gio libres
corps contrastif sur  cuda    |    latence mesurée sur  cpu


## Constantes

Les germes sont **nommés une fois** et passés partout où un tirage existe. `R = 5` : le surcoût de
l'imbrication est un facteur ≈ `K_ext`, donc ×5 reste tenable sur CPU quand ×10 doublerait un
calcul déjà lourd pour un gain de précision faible.

`k = 5` plis, tranché par [Protocole de mesure](https://github.com/AmauryTISSOT/microservice_rgpd/issues/45)
sur deux arguments indépendants : la dégénérescence des étiquettes rares à k=10, et les 8 paires
minimales qui exigent au moins k groupes. Le *label powerset* est structurellement exclu — 19
combinaisons d'étiquettes distinctes dont 9 vues une seule fois.

**La révision de l'encodeur est épinglée.** `main` sur le Hub peut bouger ; sans épinglage, deux
exécutions à six mois d'écart chargeraient deux modèles différents sans que rien ne le signale.

In [2]:
def racine_du_depot(depart=None):
    """Remonte jusqu'au dépôt, repéré par sa solution — jamais un chemin relatif au répertoire
    courant. Le notebook est lancé tantôt depuis `exploration/`, tantôt depuis la racine."""
    depart = (depart or Path.cwd()).resolve()
    for dossier in (depart, *depart.parents):
        if (dossier / "MicroserviceRgpd.slnx").is_file():
            return dossier
    raise RuntimeError(f"Racine du dépôt introuvable en remontant depuis {depart}")


RACINE = racine_du_depot()

#: Les germes. R = 5, et le critère s'applique au **pire** d'entre eux, pas à la médiane.
GERMES = [20180525, 20190523, 20200101, 20210704, 20221123]

K_EXTERNE = 5          # les plis dont sortent les prédictions hors-pli
K_INTERNE = 5          # les plis imbriqués où se règle la tête
SEUIL = 0.5            # le seuil de chaque tête, non réglé — voir la règle d'arbitrage

ENCODEUR = "intfloat/multilingual-e5-small"
#: Révision épinglée : sans elle, la reproductibilité annoncée serait creuse.
REVISION = "614241f622f53c4eeff9890bdc4f31cfecc418b3"
#: Les modèles `e5` exigent ce préfixe. L'omettre dégrade les vecteurs *sans aucun signal*.
PREFIXE_E5 = "query: "

#: Les six configurations de la tête, réglées **dans** les plis. `lbfgs` n'a aucun tirage : le
#: seul aléa des montages 1 et 2 est donc le découpage, et rien d'autre.
GRILLE_TETE = [{"C": c, "class_weight": w} for c in (0.1, 1.0, 10.0) for w in (None, "balanced")]

#: Les paramètres SetFit sont ceux **publiés par défaut**, délibérément non réglés : les régler
#: aurait ajouté une dimension à une recherche déjà imbriquée, sur 120 exemples.
SETFIT_ITERATIONS = 20
SETFIT_LOT = 16
SETFIT_EPOQUES = 1

SLUGS = ["acces", "rectification", "effacement", "limitation", "portabilite", "opposition",
         "hors-perimetre"]
HORS_PERIMETRE = "hors-perimetre"
INDICE_HP = SLUGS.index(HORS_PERIMETRE)

#: La table de correspondance du sidecar, **recopiée** : le projet d'exploration ne déclare
#: aucune dépendance sur le sidecar, pas même en `path` (#49). Il ne lit que des fichiers plats.
CANONIQUE_PAR_SLUG = {
    "acces": "Access", "rectification": "Rectification", "effacement": "Erasure",
    "limitation": "Restriction", "portabilite": "Portability", "opposition": "Objection",
    "hors-perimetre": "OutOfScope",
}
SLUG_PAR_CANONIQUE = {v: k for k, v in CANONIQUE_PAR_SLUG.items()}

#: Les cinq erreurs de `qwen3:8b` que le montage actuel signale déjà. La condition 2 du critère
#: est binaire : en perdre une seule est un « non », quel que soit le gain par ailleurs.
ERREURS_ATTRAPEES_PAR_A = {"acc-10", "hop-15", "hop-22", "lim-08", "por-08"}
#: Les deux erreurs que la règle de consensus ne peut **structurellement** pas attraper : le
#: lexique s'y trompe *avec* le LLM. Y renoncer et adopter le consensus sont la même décision.
ERREURS_INATTEIGNABLES = {"edg-04", "hop-14"}
#: Le taux de fausse alarme du montage actuel, la barre à battre.
TAUX_A = 24 / 113

print(f"racine du dépôt : {RACINE}")
print(f"germes : {GERMES}")

racine du dépôt : C:\Users\G713\repos\microservice_rgpd\.claude\worktrees\wf50
germes : [20180525, 20190523, 20200101, 20210704, 20221123]


## Les données

Trois fichiers plats, tous versionnés. Le notebook ne parle à aucun moteur servi — c'est la règle
qui départage les deux environnements d'`exploration/` : *ce qui parle à un moteur servi tourne
avec l'environnement de production, ce qui calcule sur des fichiers figés tourne avec celui de
l'exploration.*

| Rôle | Fichier |
| --- | --- |
| Vérité terrain, 120 exemples | `corpus/demandes-rgpd.fr.jsonl` |
| **L** — le témoin du lexique | `src/sidecar/tests/witness/lexicon-corpus.jsonl` |
| **V** — les avis de `qwen3:8b` | `exploration/qwen3-8b-corpus.jsonl` |

Le témoin du lexique est lu **là où il est**. Le recopier dans `exploration/` en ferait une
seconde vérité qui divergerait au premier changement de règle.

In [3]:
def charger_corpus(racine):
    with (racine / "corpus" / "demandes-rgpd.fr.jsonl").open(encoding="utf-8") as f:
        return [json.loads(ligne) for ligne in f]


def charger_avis(chemin):
    """Rend, par identifiant, l'ensemble des droits en slugs français.

    Une panne reste une panne : le sidecar traite un avis invalide comme une panne du moteur et
    jamais comme un avis faible, et l'artefact garde cette distinction. Le témoin du lexique,
    lui, ne porte ni statut ni confiance — par construction.
    """
    avis = {}
    with chemin.open(encoding="utf-8") as f:
        for ligne in f:
            ligne = json.loads(ligne)
            if ligne.get("status", 200) != 200:
                raise ValueError(f"Avis en panne dans {chemin.name} : {ligne['id']}")
            avis[ligne["id"]] = {SLUG_PAR_CANONIQUE[nom] for nom in ligne["rights"]}
    return avis


corpus = charger_corpus(RACINE)
identifiants = [ligne["id"] for ligne in corpus]
textes = [ligne["texte"] for ligne in corpus]
verite = [set(ligne["droits"]) for ligne in corpus]
Y = np.array([[1 if s in ligne["droits"] else 0 for s in SLUGS] for ligne in corpus], dtype=int)

L = [charger_avis(RACINE / "src/sidecar/tests/witness/lexicon-corpus.jsonl")[i] for i in identifiants]
V = [charger_avis(RACINE / "exploration/qwen3-8b-corpus.jsonl")[i] for i in identifiants]

N_EXEMPLES = len(corpus)
assert N_EXEMPLES == 120 and len(L) == 120 and len(V) == 120

print(f"{N_EXEMPLES} exemples")
print("étiquettes  :", dict(zip(SLUGS, Y.sum(axis=0))))
print("cardinalité :", dict(sorted(Counter(Y.sum(axis=1)).items())),
      f"→ cardinalité d'étiquetage {Y.sum() / N_EXEMPLES:.2f}")
combinaisons = Counter(tuple(sorted(d)) for d in verite)
print(f"combinaisons distinctes : {len(combinaisons)}, "
      f"dont vues une seule fois : {sum(1 for n in combinaisons.values() if n == 1)}")

120 exemples
étiquettes  : {'acces': np.int64(23), 'rectification': np.int64(14), 'effacement': np.int64(26), 'limitation': np.int64(14), 'portabilite': np.int64(13), 'opposition': np.int64(20), 'hors-perimetre': np.int64(30)}
cardinalité : {np.int64(1): 101, np.int64(2): 18, np.int64(3): 1} → cardinalité d'étiquetage 1.17
combinaisons distinctes : 19, dont vues une seule fois : 9


## Préalable de validité nº 1 — le piège des accents

L'issue [`ollama#15609`](https://github.com/ollama/ollama/issues/15609), ouverte, décrit un
`strip_accents` perdu **à la conversion GGUF** : mots accentués écrasés en `[UNK]`, et des mots
sans rapport qui se retrouvent à 0,9 de similarité. Sur un corpus français, un plongement sourd
aux accents produirait des chiffres **inexploitables sans que rien ne le signale**.

Ce défaut vise l'empaquetage pour Ollama. Ce notebook charge le modèle par `sentence-transformers`
depuis Hugging Face, donc avec le **tokeniseur d'origine** : le banc d'essai est *a priori* hors
d'atteinte. « *A priori* » n'est pas une mesure — la vérification est faite ici, parce que c'est
la première cause déclarée de « non concluant ».

**L'alarme reste entière pour la production.** Elle vise l'option « zéro dépendance via Ollama »
recensée par [Modèles et outillage](https://github.com/AmauryTISSOT/microservice_rgpd/issues/44),
et personne ne l'a testée sur les candidats sérieux. À rejouer avant tout passage en production
par cette voie.

In [4]:
from sentence_transformers import SentenceTransformer

rss_avant = psutil.Process().memory_info().rss
t0 = time.perf_counter()
encodeur = SentenceTransformer(ENCODEUR, revision=REVISION, device="cpu")
SECONDES_CHARGEMENT = time.perf_counter() - t0
RSS_MODELE = psutil.Process().memory_info().rss - rss_avant

# 1. Aucun mot accentué du corpus ne tombe en [UNK].
import re
mots_accentues = sorted({m.lower() for t in textes for m in re.findall(r"\w+", t)
                         if any(c in "àâäéèêëîïôöùûüçÀÂÄÉÈÊËÎÏÔÖÙÛÜÇ" for c in m)})
inconnu = encodeur.tokenizer.unk_token
ecrases = [m for m in mots_accentues if inconnu in encodeur.tokenizer.tokenize(m)]

# 2. L'accent doit *porter* de l'information : le vecteur d'un mot accentué et celui de sa
#    version dépouillée ne doivent pas être confondus au point que tout se vaille.
paires = [("données", "donnees"), ("résiliation", "resiliation"), ("être", "etre"),
          ("opposition à une finalité", "opposition a une finalite")]
sondes = [PREFIXE_E5 + m for paire in paires for m in paire]
sans_rapport = [PREFIXE_E5 + t for t in
                ("le train de 8 h 12 part du quai numéro trois",
                 "recette de la tarte aux pommes de ma grand-mère")]
vecteurs = encodeur.encode(sondes + sans_rapport, normalize_embeddings=True)
sim_paires = [float(vecteurs[2 * i] @ vecteurs[2 * i + 1]) for i in range(len(paires))]
sim_temoin = float(vecteurs[-2] @ vecteurs[-1])

print(f"mots accentués du corpus            : {len(mots_accentues)}")
print(f"écrasés en {inconnu:<24}: {len(ecrases)}  {ecrases if ecrases else ''}")
print(f"similarité accentué / dépouillé     : "
      f"{', '.join(f'{p[0]}≈{p[1]} {s:.3f}' for p, s in zip(paires, sim_paires))}")
print(f"similarité de deux textes étrangers : {sim_temoin:.3f}  (le plancher de cet encodeur)")

BUG_ACCENTS = bool(ecrases)
print()
print("PRÉALABLE nº 1 :", "ÉCHEC — bug d'accents confirmé, tout ce qui suit est ininterprétable"
      if BUG_ACCENTS else "franchi — le tokeniseur d'origine garde les accents entiers")

mots accentués du corpus            : 183
écrasés en <unk>                   : 0  
similarité accentué / dépouillé     : données≈donnees 0.871, résiliation≈resiliation 0.960, être≈etre 0.905, opposition à une finalité≈opposition a une finalite 0.967
similarité de deux textes étrangers : 0.791  (le plancher de cet encodeur)

PRÉALABLE nº 1 : franchi — le tokeniseur d'origine garde les accents entiers


## Le point de comparaison A — recalculé, jamais recopié

Le montage actuel du § 6.2 de la spec : `alarme = (V ≠ L)`. Les chiffres du critère #48 sont
**recalculés ici depuis les fichiers**, et non repris de leur ticket : un chiffre repris à la main
est un chiffre qui peut avoir vieilli sans qu'on le sache.

L'accord est l'**accord exact** entre ensembles de droits — c'est la notion qu'emploie le § 6.2, et
la seule qui ait un sens pour une règle d'alarme : un avis « à moitié d'accord » n'existe pas dans
le contrat.

In [5]:
accord = lambda a, b: a == b

erreurs_llm = [i for i in range(N_EXEMPLES) if not accord(V[i], verite[i])]
corrects_llm = [i for i in range(N_EXEMPLES) if accord(V[i], verite[i])]
alarme_A = [not accord(V[i], L[i]) for i in range(N_EXEMPLES)]
fausses_A = [i for i in corrects_llm if alarme_A[i]]
attrapees_A = {identifiants[i] for i in erreurs_llm if alarme_A[i]}
echappent_A = {identifiants[i] for i in erreurs_llm if not alarme_A[i]}

print(f"erreurs de qwen3:8b            : {len(erreurs_llm)} / {N_EXEMPLES}"
      f"  {sorted(identifiants[i] for i in erreurs_llm)}")
print(f"verdicts corrects (dénominateur): {len(corrects_llm)}")
print(f"alarmes de A                   : {sum(alarme_A)} / {N_EXEMPLES}"
      f" = {sum(alarme_A) / N_EXEMPLES:.1%} de déclenchement")
print(f"  dont fausses                 : {len(fausses_A)} = {len(fausses_A) / len(corrects_llm):.1%}"
      f" des verdicts corrects   ← la barre")
print(f"  erreurs déjà attrapées       : {len(attrapees_A)}  {sorted(attrapees_A)}")
print(f"  erreurs qui échappent        : {len(echappent_A)}  {sorted(echappent_A)}")
print(f"exactitude du lexique seul     : {sum(1 for i in range(N_EXEMPLES) if accord(L[i], verite[i]))}"
      f" / {N_EXEMPLES}")
print(f"violations d'I2 par le LLM     : "
      f"{sum(1 for a in V if HORS_PERIMETRE in a and len(a) > 1)}")

assert attrapees_A == ERREURS_ATTRAPEES_PAR_A, "Les 5 erreurs signalées ne sont plus les mêmes."
assert echappent_A == ERREURS_INATTEIGNABLES, "Les 2 erreurs inatteignables ne sont plus les mêmes."
assert len(fausses_A) == 24 and len(corrects_llm) == 113, "Le point de comparaison a bougé."
print("\nLe point de comparaison du critère #48 est confirmé sur les fichiers.")

erreurs de qwen3:8b            : 7 / 120  ['acc-10', 'edg-04', 'hop-14', 'hop-15', 'hop-22', 'lim-08', 'por-08']
verdicts corrects (dénominateur): 113
alarmes de A                   : 29 / 120 = 24.2% de déclenchement
  dont fausses                 : 24 = 21.2% des verdicts corrects   ← la barre
  erreurs déjà attrapées       : 5  ['acc-10', 'hop-15', 'hop-22', 'lim-08', 'por-08']
  erreurs qui échappent        : 2  ['edg-04', 'hop-14']
exactitude du lexique seul     : 94 / 120
violations d'I2 par le LLM     : 0

Le point de comparaison du critère #48 est confirmé sur les fichiers.


## Le découpage — stratification multi-label **et** contrainte de groupes

**Un trou d'outillage réel.** Aucune bibliothèque ne combine les deux, et il faut les deux :

- la **stratification** pour que les trois étiquettes à 13-14 exemples ne disparaissent pas d'un pli ;
- les **groupes** pour que les **8 paires minimales** du corpus ne se retrouvent pas à cheval sur
  la frontière apprentissage/test. Ce sont les exemples les plus discriminants — deux textes
  quasi identiques dont la qualification diffère —, et les séparer reviendrait à **fuiter la
  réponse**.

Ce découpage est donc écrit à la main : c'est la stratification itérative du premier ordre de
Sechidis et al., transposée aux groupes. L'unité distribuée n'est plus l'exemple mais le groupe,
et ses étiquettes sont la somme de celles de ses membres.

L'arbitrage entre stratification itérative simple et sa variante de second ordre est **mesurable,
pas doctrinal** — Sechidis et Szymański se contredisent frontalement sur notre régime — et ne
porterait que sur les 19 exemples multi-étiquettes, pour une cardinalité d'étiquetage de 1,17. Le
premier ordre est retenu ; l'écart est vérifié plus bas.

In [6]:
#: Les huit couples quasi identiques documentés par `corpus/README.md`.
PAIRES_MINIMALES = [
    ("edg-13", "edg-14"),   # présence ou non d'une valeur de remplacement
    ("edg-04", "edg-05"),   # question *sur* un droit ou exercice du droit
    ("eff-02", "hop-03"),   # supprimer un compte ou résilier un abonnement
    ("eff-03", "hop-24"),   # même impératif familier, objet « données » ou objet « contrat »
    ("opp-02", "mul-01"),   # l'enregistrement est préservé, ou il est aussi visé
    ("edg-11", "edg-12"),   # négation portant sur l'effacement : opposition ou limitation
    ("acc-01", "hop-22"),   # demande d'accès RGPD ou législation sectorielle
    ("eff-05", "opp-01"),   # retrait de consentement ou opposition à une finalité
]


def construire_groupes(identifiants):
    """Huit groupes de deux pour les paires minimales ; tout le reste est singleton."""
    indice = {ident: i for i, ident in enumerate(identifiants)}
    groupes = [[indice[g], indice[d]] for g, d in PAIRES_MINIMALES]
    apparies = {i for membres in groupes for i in membres}
    groupes.extend([i] for i in range(len(identifiants)) if i not in apparies)
    return groupes


def _meilleur_pli(manque_etiquette, manque_taille, tirage):
    """Le pli qui manque le plus de l'étiquette ; à égalité, celui qui manque le plus d'exemples ;
    à égalité encore, un tirage — départager par l'indice biaiserait vers les premiers plis."""
    candidats = np.flatnonzero(manque_etiquette >= manque_etiquette.max() - 1e-9)
    if len(candidats) > 1:
        second = manque_taille[candidats]
        candidats = candidats[second >= second.max() - 1e-9]
    return int(tirage.choice(candidats)) if len(candidats) > 1 else int(candidats[0])


def decouper(Y, groupes, k, germe):
    """Rend, pour chaque exemple, le numéro du pli où il est en **test**.

    À chaque tour, l'étiquette la plus rare encore à placer commande, et le groupe qui la porte va
    au pli qui en manque le plus. Les rares sont ainsi servies les premières, quand la latitude est
    maximale — c'est tout l'intérêt de la méthode sur un corpus où trois étiquettes tiennent en
    treize exemples.
    """
    tirage = np.random.default_rng(germe)
    etiquettes_groupe = np.array([Y[membres].sum(axis=0) for membres in groupes])
    tailles_groupe = np.array([len(membres) for membres in groupes])

    manque = np.tile(etiquettes_groupe.sum(axis=0) / k, (k, 1)).astype(float)
    manque_taille = np.full(k, tailles_groupe.sum() / k, dtype=float)

    restants = list(tirage.permutation(len(groupes)))
    affectation = np.full(len(groupes), -1, dtype=int)

    while restants:
        en_attente = etiquettes_groupe[restants].sum(axis=0)
        positives = [l for l in range(Y.shape[1]) if en_attente[l] > 0]
        if positives:
            rare = min(positives, key=lambda l: (en_attente[l], l))
            candidats = [g for g in restants if etiquettes_groupe[g][rare] > 0]
        else:
            rare, candidats = None, list(restants)

        for groupe in candidats:
            colonne = manque_taille if rare is None else manque[:, rare]
            pli = _meilleur_pli(colonne, manque_taille, tirage)
            affectation[groupe] = pli
            manque[pli] -= etiquettes_groupe[groupe]
            manque_taille[pli] -= tailles_groupe[groupe]
            restants.remove(groupe)

    plis = np.empty(Y.shape[0], dtype=int)
    for groupe, membres in enumerate(groupes):
        plis[membres] = affectation[groupe]
    return plis


GROUPES = construire_groupes(identifiants)
print(f"{len(GROUPES)} groupes pour {N_EXEMPLES} exemples "
      f"({sum(1 for g in GROUPES if len(g) > 1)} paires minimales, le reste en singletons)")

112 groupes pour 120 exemples (8 paires minimales, le reste en singletons)


### Condition de validité nº 2 — le découpage est-il praticable ?

Trois causes déclarées de « non concluant » se constatent ici, **avant toute mesure** : plis
dégénérés, paires minimales inséparables, étiquette absente d'un pli d'apprentissage. Une
étiquette absente d'un pli de **test** n'est pas une invalidité — c'est une perte de puissance sur
cette étiquette, à signaler.

In [7]:
def verifier_decoupage(Y, plis, groupes, k):
    tailles = [int((plis == f).sum()) for f in range(k)]
    coupees = [[identifiants[i] for i in m] for m in groupes if len({plis[i] for i in m}) > 1]
    absente_apprentissage, absente_test = {}, {}
    for f in range(k):
        manquantes_a = [SLUGS[l] for l in range(Y.shape[1]) if Y[plis != f].sum(axis=0)[l] == 0]
        manquantes_t = [SLUGS[l] for l in range(Y.shape[1]) if Y[plis == f].sum(axis=0)[l] == 0]
        if manquantes_a:
            absente_apprentissage[f] = manquantes_a
        if manquantes_t:
            absente_test[f] = manquantes_t
    return {
        "tailles": tailles,
        "paires_coupees": coupees,
        "etiquette_absente_en_apprentissage": absente_apprentissage,
        "etiquette_absente_en_test": absente_test,
        "valide": not coupees and not absente_apprentissage and min(tailles) > 0,
    }


PLIS = {germe: decouper(Y, GROUPES, K_EXTERNE, germe) for germe in GERMES}
DECOUPAGE_VALIDE = True
for germe, plis in PLIS.items():
    bilan = verifier_decoupage(Y, plis, GROUPES, K_EXTERNE)
    DECOUPAGE_VALIDE &= bilan["valide"]
    repartition = {SLUGS[l]: [int(Y[plis == f].sum(axis=0)[l]) for f in range(K_EXTERNE)]
                   for l in range(len(SLUGS))}
    print(f"germe {germe} | tailles {bilan['tailles']} | "
          f"{'valide' if bilan['valide'] else 'INVALIDE ' + str(bilan)}")
    print(f"             rares par pli : "
          + "  ".join(f"{s} {repartition[s]}" for s in ("rectification", "limitation", "portabilite")))
    if bilan["etiquette_absente_en_test"]:
        print(f"             ⚠ absente d'un pli de test : {bilan['etiquette_absente_en_test']}")

print()
print("CONDITION nº 2 :", "franchie — aucun pli dégénéré, aucune paire minimale coupée"
      if DECOUPAGE_VALIDE else "ÉCHEC — validation croisée impraticable")

germe 20180525 | tailles [23, 25, 25, 23, 24] | valide
             rares par pli : rectification [2, 3, 3, 3, 3]  limitation [3, 2, 3, 3, 3]  portabilite [3, 3, 3, 2, 2]
germe 20190523 | tailles [25, 23, 25, 25, 22] | valide
             rares par pli : rectification [3, 2, 3, 3, 3]  limitation [3, 2, 3, 3, 3]  portabilite [2, 3, 3, 2, 3]
germe 20200101 | tailles [24, 23, 23, 24, 26] | valide
             rares par pli : rectification [3, 2, 3, 3, 3]  limitation [3, 3, 3, 2, 3]  portabilite [2, 3, 2, 3, 3]
germe 20210704 | tailles [24, 26, 25, 22, 23] | valide
             rares par pli : rectification [3, 3, 3, 3, 2]  limitation [3, 3, 3, 3, 2]  portabilite [2, 3, 3, 2, 3]
germe 20221123 | tailles [22, 25, 27, 24, 22] | valide
             rares par pli : rectification [3, 2, 3, 3, 3]  limitation [3, 3, 3, 3, 2]  portabilite [3, 3, 3, 2, 2]

CONDITION nº 2 : franchie — aucun pli dégénéré, aucune paire minimale coupée


## La tête et la règle de décision

**Tête commune aux trois montages** : `OneVsRestClassifier(LogisticRegression(solver="lbfgs"))`.
Si la tête changeait entre les montages, on ne mesurerait plus la représentation mais un mélange.
`lbfgs` est déterministe et n'a aucun tirage : toute variance observée entre les montages 1 et 2
est donc imputable à la représentation, à rien d'autre.

### L'exclusivité de `hors-perimetre`, appliquée **par construction**

Le § 5.4 de la spec traite un avis invalide comme une **panne**, jamais comme un avis faible : un
témoin qui rendrait `{OutOfScope, Access}` serait rejeté à la frontière du sidecar, donc
**absent** — et un témoin absent fait tomber la règle de consensus sur laquelle repose tout le
critère. L'invariant est donc appliqué **dans le modèle**, à l'intérieur de la validation croisée,
jamais en rattrapage après coup : sans quoi les prédictions hors-pli ne décriraient pas ce qu'on
déploierait.

Règle d'arbitrage, **figée par #47 et jamais optimisée dans les plis** :

> Si `hors-perimetre` passe son seuil **et** qu'au moins un droit passe le sien, garder celui des
> deux camps dont la probabilité est la plus haute. Si aucune étiquette ne passe son seuil, rendre
> `hors-perimetre`.

C'est la seule forme qui rende le **taux de violation brute mesurable** : en exclusivité
structurelle, il serait nul par construction et le notebook n'apprendrait rien.

**Assumé** : cette règle est arbitraire à la marge et c'est un hyperparamètre déguisé. Elle est
figée une fois pour toutes pour qu'elle ne se paie pas en sur-ajustement, et le taux de violation
brute est rendu à côté.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer


def ajuster_tete(X, Y_, configuration):
    tete = OneVsRestClassifier(LogisticRegression(
        solver="lbfgs", max_iter=2000, C=configuration["C"],
        class_weight=configuration["class_weight"]))
    tete.fit(X, Y_)
    return tete


def decider(probabilites):
    """La règle d'arbitrage figée. Rend aussi ce qu'*aurait* été la sortie brute, sans quoi le
    taux de violation d'I2 avant contrainte ne serait pas observable."""
    franchissent = [l for l in range(len(SLUGS)) if probabilites[l] >= SEUIL]
    droits = [l for l in franchissent if l != INDICE_HP]
    hp_franchit = INDICE_HP in franchissent
    violation = hp_franchit and bool(droits)

    if hp_franchit and droits:
        predit = ([INDICE_HP] if probabilites[INDICE_HP] > max(probabilites[l] for l in droits)
                  else droits)
    elif hp_franchit or not droits:
        predit = [INDICE_HP]
    else:
        predit = droits

    triees = np.sort(probabilites)[::-1]
    return {
        "brut": [SLUGS[l] for l in sorted(franchissent)],
        "predit": [SLUGS[l] for l in sorted(predit)],
        # Score **continu**, jamais ordinal : on peut toujours grossir un continu en trois
        # degrés, jamais l'inverse.
        "marge": float(triees[0] - triees[1]) if franchissent else 0.0,
        "aucun_seuil_franchi": not franchissent,
        "violation_i2_brute": violation,
    }


def macro_f1(reference, predit):
    """Moyenne **non pondérée** sur les sept étiquettes, calculée sur la sortie *contrainte* —
    c'est-à-dire sur ce qu'on déploierait. Le macro donne aux trois étiquettes rares le poids que
    le micro leur refuserait, et c'est sur elles que le témoin est fragile."""
    scores = []
    for slug in SLUGS:
        vp = sum(1 for r, p in zip(reference, predit) if slug in r and slug in p)
        fp = sum(1 for r, p in zip(reference, predit) if slug not in r and slug in p)
        fn = sum(1 for r, p in zip(reference, predit) if slug in r and slug not in p)
        scores.append(0.0 if vp == 0 else 2 * vp / (2 * vp + fp + fn))
    return float(np.mean(scores))


def wilson(succes, total, z=1.959963984540054):
    """Intervalle binomial de Wilson — l'instrument d'incertitude retenu par #48, et non
    « moyenne ± écart-type entre plis », dont #45 rappelle que ce n'est pas un intervalle de
    confiance (théorème d'impossibilité de Bengio & Grandvalet)."""
    if total == 0:
        return (0.0, 1.0)
    p = succes / total
    d = 1 + z ** 2 / total
    centre = (p + z ** 2 / (2 * total)) / d
    demi = z * math.sqrt(p * (1 - p) / total + z ** 2 / (4 * total ** 2)) / d
    return (max(0.0, centre - demi), min(1.0, centre + demi))

## La boucle de mesure

**Le réglage de la tête est imbriqué dans les plis.** C'est la propriété qui rend les chiffres
lisibles, et la seule qui soit facile à casser par distraction : la configuration retenue pour un
pli externe est choisie sur une validation croisée **interne** de son seul jeu d'apprentissage. Le
pli de test externe n'intervient à aucun moment du choix.

Le découpage interne respecte lui aussi les groupes : une paire minimale coupée entre
apprentissage et validation interne fausserait le réglage exactement comme elle fausserait la
mesure.

Six configurations × 5 plis internes × 5 plis externes, plus le réajustement final par pli
externe : **155 ajustements de tête par montage et par germe**.

In [9]:
def _groupes_locaux(indices_apprentissage):
    """Réindexe les groupes dans le repère du seul jeu d'apprentissage."""
    groupe_de = {i: g for g, membres in enumerate(GROUPES) for i in membres}
    local = {int(i): j for j, i in enumerate(indices_apprentissage)}
    par_groupe = {}
    for i in indices_apprentissage:
        par_groupe.setdefault(groupe_de[int(i)], []).append(local[int(i)])
    return list(par_groupe.values())


def regler_puis_ajuster(X, Y_, groupes, germe):
    """Choisit la configuration sur des plis **internes**, puis réajuste sur tout l'apprentissage."""
    internes = decouper(Y_, groupes, K_INTERNE, germe + 1)
    meilleure, meilleur_score = None, -1.0
    for configuration in GRILLE_TETE:
        predits, references = [], []
        for f in range(K_INTERNE):
            appr, val = internes != f, internes == f
            if Y_[appr].sum(axis=0).min() == 0:
                continue          # une étiquette absente rend cette configuration inévaluable ici
            tete = ajuster_tete(X[appr], Y_[appr], configuration)
            P = np.asarray(tete.predict_proba(X[val]), dtype=float)
            predits += [set(decider(p)["predit"]) for p in P]
            references += [{SLUGS[l] for l in np.flatnonzero(r)} for r in Y_[val]]
        score = macro_f1(references, predits) if predits else -1.0
        if score > meilleur_score + 1e-12:
            meilleure, meilleur_score = configuration, score
    if meilleure is None:
        raise RuntimeError("Aucune configuration n'a pu être évaluée : réglage qui ne converge pas.")
    return ajuster_tete(X, Y_, meilleure), meilleure, meilleur_score


def mesurer(montage, representation, germes=GERMES, journal=True):
    """Rend une prédiction **hors-pli** par exemple et par germe.

    `representation(indices_apprentissage, germe, pli)` rend la matrice des 120 exemples. Elle
    reçoit les indices d'apprentissage parce que certaines représentations s'ajustent — le
    vocabulaire TF-IDF, l'encodeur SetFit — et qu'un ajustement sur les 120 serait une fuite.
    """
    enregistrements = []
    for germe in germes:
        plis = PLIS[germe]
        for f in range(K_EXTERNE):
            debut = time.perf_counter()
            appr = np.flatnonzero(plis != f)
            test = np.flatnonzero(plis == f)
            X = representation(appr, germe, f)
            tete, configuration, score = regler_puis_ajuster(
                X[appr], Y[appr], _groupes_locaux(appr), germe)
            P = np.asarray(tete.predict_proba(X[test]), dtype=float)
            for rang, i in enumerate(test):
                enregistrements.append({
                    "montage": montage, "germe": germe, "pli": f, "id": identifiants[int(i)],
                    "probabilites": {s: round(float(P[rang][l]), 6) for l, s in enumerate(SLUGS)},
                    "reglage": configuration, **decider(P[rang]),
                })
            if journal:
                print(f"  {montage} germe {germe} pli {f} : {configuration} "
                      f"macro-F1 interne {score:.3f} — {time.perf_counter() - debut:.1f} s",
                      flush=True)
    return enregistrements


ENREGISTREMENTS = {}
DUREES = {}

## Montage 1 — `TfidfVectorizer`, la ligne de base

Ce que l'automatisation du lexique donnerait. TF-IDF n'est pas le candidat : c'est le **même
matériau** que le lexique — de la surface lexicale — et il se trompera donc probablement là où le
lexique se trompe. Il répond à une question réelle et bon marché : *le plongement dense fait-il
mieux que le sac de mots sur ce corpus ?* Si `gelé ≈ TF-IDF`, c'est le résultat majeur du
notebook.

Le fait le plus dur contre lui, relevé par [État de l'art](https://github.com/AmauryTISSOT/microservice_rgpd/issues/43) :
**755 types de vocabulaire dont 65 % de hapax**, et 77 seulement vus dans au moins 5 documents.

Le vocabulaire est ajusté **sur le seul jeu d'apprentissage** de chaque pli : l'ajuster sur les 120
ferait entrer le vocabulaire du test dans la représentation.

In [10]:
def representation_tfidf(appr, germe, pli):
    vectoriseur = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), sublinear_tf=True, min_df=1)
    vectoriseur.fit([textes[int(i)] for i in appr])
    return vectoriseur.transform(textes)


t0 = time.perf_counter()
ENREGISTREMENTS["tfidf"] = mesurer("tfidf", representation_tfidf)
DUREES["tfidf"] = time.perf_counter() - t0
print(f"\nmontage 1 : {DUREES['tfidf']:.1f} s")

  tfidf germe 20180525 pli 0 : {'C': 1.0, 'class_weight': 'balanced'} macro-F1 interne 0.308 — 2.2 s


  tfidf germe 20180525 pli 1 : {'C': 10.0, 'class_weight': 'balanced'} macro-F1 interne 0.239 — 2.2 s


  tfidf germe 20180525 pli 2 : {'C': 1.0, 'class_weight': 'balanced'} macro-F1 interne 0.281 — 2.2 s


  tfidf germe 20180525 pli 3 : {'C': 0.1, 'class_weight': 'balanced'} macro-F1 interne 0.244 — 2.3 s


  tfidf germe 20180525 pli 4 : {'C': 0.1, 'class_weight': 'balanced'} macro-F1 interne 0.255 — 2.1 s


...

  tfidf germe 20221123 pli 1 : {'C': 0.1, 'class_weight': 'balanced'} macro-F1 interne 0.293 — 2.3 s


  tfidf germe 20221123 pli 2 : {'C': 1.0, 'class_weight': 'balanced'} macro-F1 interne 0.196 — 2.2 s


  tfidf germe 20221123 pli 3 : {'C': 0.1, 'class_weight': 'balanced'} macro-F1 interne 0.258 — 2.2 s


  tfidf germe 20221123 pli 4 : {'C': 0.1, 'class_weight': 'balanced'} macro-F1 interne 0.224 — 2.1 s



montage 1 : 54.8 s


## Montage 2 — `multilingual-e5-small` gelé, le candidat principal

L'encodeur ne bouge pas : les 120 textes sont plongés **une fois**, et seule la tête apprend. C'est
la seule des quatre familles retenues par #43 dont la représentation soit structurellement d'une
**autre nature** que celle du témoin déjà en place — et c'est précisément ce qu'un second témoin
doit apporter.

Le préfixe `query: ` est obligatoire pour les modèles `e5`. L'omettre dégrade les vecteurs **sans
aucun signal**.

L'encodeur est chargé **sur CPU** et y reste : c'est l'objet même dont la cellule de coût
mesurera la latence, et le mesurer ailleurs que là où le sidecar le ferait tourner n'aurait aucun
sens. Seul le corps contrastif du montage 3 va sur la carte, et il y va parce qu'on l'*entraîne*.

Geler l'encodeur permet de le plonger une seule fois hors de la boucle : la représentation
n'apprend rien du jeu d'apprentissage, donc l'encoder en dehors des plis n'est pas une fuite —
c'est exactement ce que « gelé » veut dire.

In [11]:
PLONGEMENTS_GELES = encodeur.encode([PREFIXE_E5 + t for t in textes],
                                    normalize_embeddings=True, batch_size=16)
print(f"plongements : {PLONGEMENTS_GELES.shape}")

t0 = time.perf_counter()
ENREGISTREMENTS["e5-gele"] = mesurer("e5-gele", lambda appr, germe, pli: PLONGEMENTS_GELES)
DUREES["e5-gele"] = time.perf_counter() - t0
print(f"\nmontage 2 : {DUREES['e5-gele']:.1f} s")

plongements : (120, 384)


  e5-gele germe 20180525 pli 0 : {'C': 10.0, 'class_weight': 'balanced'} macro-F1 interne 0.636 — 1.9 s


  e5-gele germe 20180525 pli 1 : {'C': 10.0, 'class_weight': 'balanced'} macro-F1 interne 0.630 — 1.9 s


  e5-gele germe 20180525 pli 2 : {'C': 10.0, 'class_weight': 'balanced'} macro-F1 interne 0.659 — 1.9 s


  e5-gele germe 20180525 pli 3 : {'C': 10.0, 'class_weight': 'balanced'} macro-F1 interne 0.621 — 1.9 s


...

  e5-gele germe 20221123 pli 1 : {'C': 10.0, 'class_weight': 'balanced'} macro-F1 interne 0.645 — 1.9 s


  e5-gele germe 20221123 pli 2 : {'C': 10.0, 'class_weight': 'balanced'} macro-F1 interne 0.635 — 1.9 s


  e5-gele germe 20221123 pli 3 : {'C': 1.0, 'class_weight': 'balanced'} macro-F1 interne 0.631 — 1.9 s


  e5-gele germe 20221123 pli 4 : {'C': 10.0, 'class_weight': 'balanced'} macro-F1 interne 0.694 — 1.9 s



montage 2 : 47.3 s


## Montage 3 — SetFit, l'échelon d'engagement supérieur

Le montage 2 **moins l'entraînement contrastif** : même encodeur, même tête, même code de mesure,
mêmes plis. C'est ce qui rend le résultat interprétable.

- `gelé ≈ SetFit` → l'entraînement contrastif ne paie pas, on garde le montage déterministe ;
- `SetFit ≫ gelé` → l'engagement se justifie, on assume la graine ;
- `gelé ≈ TF-IDF` → le plongement dense n'apporte rien sur ce corpus.

SetFit n'était pas le candidat naturel : sa variance inter-graines mesurée jusqu'à **±11,8 points**
sur jeu déséquilibré s'ajouterait aux ±20 à 25 points d'intervalle que #45 annonce déjà sur les
étiquettes rares. Il entre comme échelon mesuré à côté des deux autres, pas comme le montage dont
dépend la conclusion.

**L'encodeur est réentraîné à chaque pli externe**, sur le seul jeu d'apprentissage de ce pli. Le
réentraîner une fois sur les 120 aurait fait entrer les textes de test dans la représentation — la
fuite la plus facile à commettre de tout ce notebook.

### Un écart au protocole, déclaré

Le corps contrastif est entraîné **une fois par pli externe**, et le réglage de la tête se fait
ensuite sur les plongements qu'il produit. Le corps a donc vu les exemples qui servent de
validation *interne* au réglage. Le réentraîner à chaque pli interne porterait le compte à
**125 entraînements** au lieu de 25 — de l'ordre de 2 h 45 sur cette carte, et de 27 heures sur ce
CPU. Le passage au GPU a divisé la durée par plus de dix sans rendre l'imbrication complète
raisonnable : l'écart tient toujours, avec un motif chiffré plus petit.

**Ce que cela n'affecte pas** : le pli de **test externe** n'entre à aucun moment, ni dans le corps
ni dans la tête. Les prédictions hors-pli restent donc propres, et ce sont elles seules dont le
critère se nourrit. L'écart ne touche que la *qualité du choix* d'hyperparamètre, pas la validité
de la mesure.

### La durée, que personne ne connaissait

Aucune carte de modèle candidate ne publie de chiffre d'inférence — encore moins d'entraînement —
sur CPU (#44). Cette cellule est la mesure, et elle la rend pour **les deux appareils** : la durée
CPU mesurée avant la bascule était de **832 s par corps**, soit environ 5 h 45 pour les 25, avec
les huit cœurs physiques saturés d'un bout à l'autre — au point de rendre la machine inutilisable.
C'est cette mesure, et non une préférence, qui a fait passer l'entraînement sur la carte.

Comptez un peu plus d'une **demi-heure** avec un GPU, plusieurs heures sans.

In [12]:
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments
import logging, tempfile, datasets

datasets.disable_progress_bars()
logging.getLogger("setfit").setLevel(logging.ERROR)

# SetFit écrit ses points de reprise dans `checkpoints/` du répertoire courant par défaut. Le
# banc d'essai ne reprend rien : les envoyer au temporaire système évite de salir le dépôt.
SORTIE_SETFIT = tempfile.mkdtemp(prefix="setfit-banc-")

_plongements_setfit = {}
DUREES_CORPS = []
VRAM_CRETE = []


def representation_setfit(appr, germe, pli):
    """Réentraîne le corps sur le seul jeu d'apprentissage du pli, puis plonge les 120 textes."""
    cle = (germe, pli)
    if cle in _plongements_setfit:
        return _plongements_setfit[cle]

    random.seed(germe)
    np.random.seed(germe % 2 ** 32)
    torch.manual_seed(germe)
    if APPAREIL_ENTRAINEMENT == "cuda":
        torch.cuda.manual_seed_all(germe)
        torch.cuda.reset_peak_memory_stats()

    modele = SetFitModel.from_pretrained(
        ENCODEUR, revision=REVISION, multi_target_strategy="one-vs-rest",
        device=APPAREIL_ENTRAINEMENT)
    X = [PREFIXE_E5 + textes[int(i)] for i in appr]
    Y_ = [[int(v) for v in Y[int(i)]] for i in appr]
    arguments = TrainingArguments(batch_size=SETFIT_LOT, num_epochs=SETFIT_EPOQUES,
                                  num_iterations=SETFIT_ITERATIONS, seed=germe,
                                  output_dir=SORTIE_SETFIT, report_to="none")
    debut = time.perf_counter()
    Trainer(model=modele, args=arguments,
            train_dataset=Dataset.from_dict({"text": X, "label": Y_})
            ).train_embeddings(x_train=X, y_train=Y_, args=arguments)
    # Les lancements CUDA sont asynchrones : sans cette barrière, le chronomètre mesurerait le
    # temps mis à *soumettre* le travail, pas celui mis à le faire.
    if APPAREIL_ENTRAINEMENT == "cuda":
        torch.cuda.synchronize()
        VRAM_CRETE.append(torch.cuda.max_memory_allocated())
    duree = time.perf_counter() - debut
    DUREES_CORPS.append(duree)
    print(f"  corps contrastif germe {germe} pli {pli} : {duree / 60:.1f} min "
          f"({len(DUREES_CORPS)}/{len(GERMES) * K_EXTERNE})", flush=True)

    _plongements_setfit[cle] = modele.model_body.encode(
        [PREFIXE_E5 + t for t in textes], normalize_embeddings=True, batch_size=16)
    del modele
    # Vingt-cinq corps chargés à la file remplissent la carte si on ne rend jamais rien : le cache
    # d'allocation de `torch` garde la mémoire libérée jusqu'à ce qu'on la lui redemande.
    if APPAREIL_ENTRAINEMENT == "cuda":
        torch.cuda.empty_cache()
    return _plongements_setfit[cle]


t0 = time.perf_counter()
ENREGISTREMENTS["setfit"] = mesurer("setfit", representation_setfit)
DUREES["setfit"] = time.perf_counter() - t0
print(f"\nmontage 3 : {DUREES['setfit'] / 60:.1f} min au total")
print(f"corps contrastif ({APPAREIL_ENTRAINEMENT}) : médiane {np.median(DUREES_CORPS) / 60:.1f} min, "
      f"total {sum(DUREES_CORPS) / 60:.1f} min sur {len(DUREES_CORPS)} entraînements")
if VRAM_CRETE:
    print(f"VRAM de crête d'un corps : {max(VRAM_CRETE) / 1024 ** 3:.2f} Gio")

{'embedding_loss': 0.2034, 'grad_norm': 0.29571324586868286, 'learning_rate': 0.0, 'epoch': 0.00411522633744856}


{'embedding_loss': 0.2813, 'grad_norm': 1.4312825202941895, 'learning_rate': 1.779816513761468e-05, 'epoch': 0.205761316872428}


{'embedding_loss': 0.1839, 'grad_norm': 0.9229796528816223, 'learning_rate': 1.3211009174311929e-05, 'epoch': 0.411522633744856}


{'embedding_loss': 0.1331, 'grad_norm': 1.3917940855026245, 'learning_rate': 8.623853211009175e-06, 'epoch': 0.6172839506172839}


{'embedding_loss': 0.0943, 'grad_norm': 1.4801268577575684, 'learning_rate': 4.036697247706423e-06, 'epoch': 0.823045267489712}


...

{'embedding_loss': 0.1467, 'grad_norm': 1.268247127532959, 'learning_rate': 8.727272727272728e-06, 'epoch': 0.6122448979591837}


{'embedding_loss': 0.1195, 'grad_norm': 0.9047028422355652, 'learning_rate': 4.181818181818182e-06, 'epoch': 0.8163265306122449}


{'train_runtime': 98.3783, 'train_samples_per_second': 39.846, 'train_steps_per_second': 2.49, 'train_loss': 0.17064358409570188, 'epoch': 1.0}
  corps contrastif germe 20221123 pli 4 : 1.7 min (25/25)


  setfit germe 20221123 pli 4 : {'C': 10.0, 'class_weight': 'balanced'} macro-F1 interne 0.933 — 107.2 s



montage 3 : 35.9 min au total
corps contrastif (cuda) : médiane 1.2 min, total 32.7 min sur 25 entraînements
VRAM de crête d'un corps : 4.13 Gio


## Condition de validité nº 3 — les prédictions ne sont ni constantes ni dégénérées

Un témoin qui répondrait toujours la même chose passerait certaines conditions du critère par
accident. C'est la dernière cause déclarée de « non concluant ».

In [13]:
PREDICTIONS_NON_DEGENEREES = True
for montage, enregistrements in ENREGISTREMENTS.items():
    for germe in GERMES:
        sorties = [tuple(r["predit"]) for r in enregistrements if r["germe"] == germe]
        distinctes = len(set(sorties))
        if distinctes <= 1:
            PREDICTIONS_NON_DEGENEREES = False
            print(f"⚠ {montage} germe {germe} : prédiction constante")
    distinctes = Counter(tuple(r["predit"]) for r in enregistrements
                         if r["germe"] == GERMES[0])
    print(f"{montage:9s} germe {GERMES[0]} : {len(distinctes)} sorties distinctes, "
          f"la plus fréquente {distinctes.most_common(1)[0][1]}/{N_EXEMPLES}")

print()
print("CONDITION nº 3 :", "franchie" if PREDICTIONS_NON_DEGENEREES
      else "ÉCHEC — prédictions constantes ou dégénérées")

tfidf     germe 20180525 : 12 sorties distinctes, la plus fréquente 68/120
e5-gele   germe 20180525 : 17 sorties distinctes, la plus fréquente 36/120
setfit    germe 20180525 : 16 sorties distinctes, la plus fréquente 31/120

CONDITION nº 3 : franchie


## L'artefact de prédictions

Les prédictions par **montage × germe × pli** deviennent un JSONL versionné, de même nature et de
même justification que `qwen3-8b-corpus.jsonl`, dont il partage le dossier.

L'effet recherché : [Lire les chiffres du notebook et prononcer le verdict](https://github.com/AmauryTISSOT/microservice_rgpd/issues/52)
**recalcule les chiffres du verdict en secondes** au lieu de rejouer 25 entraînements SetFit. Le
protocole de #45 et le critère de #48 sont des agrégations de ces prédictions : les avoir sous
forme de donnée rend le verdict **vérifiable indépendamment du notebook qui le produit**.

Contrepartie, à ne pas oublier : toute reprise de ce notebook doit réécrire cet artefact
honnêtement, faute de quoi il décrirait un montage que personne ne fait tourner.

In [14]:
CHEMIN_PREDICTIONS = RACINE / "exploration" / "temoin-predictions.jsonl"
with CHEMIN_PREDICTIONS.open("w", encoding="utf-8", newline="\n") as f:
    for montage in ("tfidf", "e5-gele", "setfit"):
        for r in ENREGISTREMENTS[montage]:
            f.write(json.dumps(r, ensure_ascii=False, sort_keys=True) + "\n")

lignes = sum(len(v) for v in ENREGISTREMENTS.values())
print(f"{CHEMIN_PREDICTIONS.relative_to(RACINE)} : {lignes} lignes "
      f"({len(ENREGISTREMENTS)} montages × {len(GERMES)} germes × {N_EXEMPLES} exemples)")

exploration\temoin-predictions.jsonl : 1800 lignes (3 montages × 5 germes × 120 exemples)


## Le critère, appliqué

Pour chaque montage et chaque germe : le taux de fausse alarme de **B2** sur les 113 verdicts
corrects, son intervalle de Wilson, et les 5 erreurs conservées.

Rappel de ce qui se joue : `qwen3:8b` ne fait que **7 erreurs sur 120** et le montage actuel en
attrape déjà **5**. Le rappel d'alarme ne peut donc **rien décider** — un intervalle de Wilson sur
« 1 attrapé sur 2 » va de 0,09 à 0,91. Le seul axe où 120 exemples ont de la puissance est celui
des **24 fausses alarmes**.

In [15]:
def evaluer(enregistrements, montage):
    resultats = {}
    for germe in GERMES:
        par_id = {r["id"]: set(r["predit"]) for r in enregistrements
                  if r["montage"] == montage and r["germe"] == germe}
        assert len(par_id) == N_EXEMPLES, "Prédictions hors-pli incomplètes."
        Nn = [par_id[i] for i in identifiants]

        alarme_B2 = [alarme_A[i] and not accord(V[i], Nn[i]) for i in range(N_EXEMPLES)]
        fausses = [i for i in corrects_llm if alarme_B2[i]]
        conservees = {identifiants[i] for i in erreurs_llm if alarme_B2[i]}
        bas, haut = wilson(len(fausses), len(corrects_llm))

        resultats[germe] = {
            "fausses_alarmes": len(fausses),
            "taux": len(fausses) / len(corrects_llm),
            "wilson": (bas, haut),
            "supprimees": len(fausses_A) - len(fausses),
            "erreurs_conservees": conservees,
            "condition_1": haut < TAUX_A,
            "condition_2": ERREURS_ATTRAPEES_PAR_A <= conservees,
            "alarmes": sum(alarme_B2),
        }
    return resultats


CRITERE = {m: evaluer(e, m) for m, e in ENREGISTREMENTS.items()}

for montage, par_germe in CRITERE.items():
    print(f"\n{montage}")
    print(f"  {'germe':>9} {'alarmes':>8} {'FA':>4} {'taux':>7} {'Wilson 95 %':>16} "
          f"{'suppr.':>7} {'5 erreurs':>10}  cond.1  cond.2")
    for germe, r in par_germe.items():
        print(f"  {germe:>9} {r['alarmes']:>8} {r['fausses_alarmes']:>4} {r['taux']:>6.1%} "
              f"  [{r['wilson'][0]:>5.1%} – {r['wilson'][1]:>5.1%}] {r['supprimees']:>7} "
              f"{len(r['erreurs_conservees'] & ERREURS_ATTRAPEES_PAR_A):>6}/5      "
              f"{'oui' if r['condition_1'] else 'NON':>3}     {'oui' if r['condition_2'] else 'NON':>3}")
        perdues = ERREURS_ATTRAPEES_PAR_A - r["erreurs_conservees"]
        if perdues:
            print(f"            erreurs perdues : {sorted(perdues)}")
    pire = max(par_germe.values(), key=lambda r: r["fausses_alarmes"])
    print(f"  pire germe : {pire['fausses_alarmes']} fausses alarmes, "
          f"borne haute {pire['wilson'][1]:.1%} contre {TAUX_A:.1%} pour A")


tfidf
      germe  alarmes   FA    taux      Wilson 95 %  suppr.  5 erreurs  cond.1  cond.2
   20180525       24   20  17.7%   [11.8% – 25.8%]       4      4/5      NON     NON
            erreurs perdues : ['acc-10']
   20190523       24   19  16.8%   [11.0% – 24.8%]       5      5/5      NON     oui
   20200101       23   18  15.9%   [10.3% – 23.8%]       6      5/5      NON     oui
   20210704       23   18  15.9%   [10.3% – 23.8%]       6      5/5      NON     oui
   20221123       27   22  19.5%   [13.2% – 27.7%]       2      5/5      NON     oui
  pire germe : 22 fausses alarmes, borne haute 27.7% contre 21.2% pour A

e5-gele
      germe  alarmes   FA    taux      Wilson 95 %  suppr.  5 erreurs  cond.1  cond.2
   20180525       18   15  13.3%   [ 8.2% – 20.8%]       9      3/5      oui     NON
            erreurs perdues : ['acc-10', 'hop-22']
   20190523       20   17  15.0%   [ 9.6% – 22.8%]       7      3/5      NON     NON
            erreurs perdues : ['acc-10', 'hop-22']
 

## Les deux comparaisons actées par la carte

*Lexique + nouveau contre lexique seul* est le critère lui-même, ci-dessus. *Nouveau seul contre
lexique seul* est la comparaison de **remplacement** : elle reste **informative** et sera rapportée,
mais elle ne pèse sur aucun verdict. Si le nouveau témoin dominait le lexique sur tous les axes,
cela alimenterait la brume « le sort du lexique » — que la carte ne tranche pas.

In [16]:
def alarme_avec(temoin):
    """La règle binaire du § 6.2 appliquée à un témoin quelconque : alarme = (V ≠ témoin)."""
    alarme = [not accord(V[i], temoin[i]) for i in range(N_EXEMPLES)]
    fausses = sum(1 for i in corrects_llm if alarme[i])
    attrapees = {identifiants[i] for i in erreurs_llm if alarme[i]}
    return sum(alarme), fausses, attrapees


print(f"{'témoin':<24} {'exactitude':>10} {'alarmes':>8} {'fausses':>8} {'erreurs attrapées':>18}")
declenche, fausses, attrapees = alarme_avec(L)
print(f"{'lexique seul (= A)':<24} "
      f"{sum(1 for i in range(N_EXEMPLES) if accord(L[i], verite[i])):>7}/120 "
      f"{declenche:>8} {fausses:>8} {len(attrapees):>15}/7")

for montage in ENREGISTREMENTS:
    for germe in (GERMES[0],):
        par_id = {r["id"]: set(r["predit"]) for r in ENREGISTREMENTS[montage] if r["germe"] == germe}
        Nn = [par_id[i] for i in identifiants]
        declenche, fausses, attrapees = alarme_avec(Nn)
        print(f"{montage + ' seul (germe 1)':<24} "
              f"{sum(1 for i in range(N_EXEMPLES) if accord(Nn[i], verite[i])):>7}/120 "
              f"{declenche:>8} {fausses:>8} {len(attrapees):>15}/7")

print()
print("exactitude du nouveau témoin, tous germes (accord exact sur les 120) :")
for montage, enregistrements in ENREGISTREMENTS.items():
    exactitudes = []
    for germe in GERMES:
        par_id = {r["id"]: set(r["predit"]) for r in enregistrements if r["germe"] == germe}
        exactitudes.append(sum(1 for i, ident in enumerate(identifiants)
                               if par_id[ident] == verite[i]))
    print(f"  {montage:9s} {exactitudes}  → min {min(exactitudes)}, max {max(exactitudes)}")

témoin                   exactitude  alarmes  fausses  erreurs attrapées
lexique seul (= A)            94/120       29       24               5/7
tfidf seul (germe 1)          49/120       72       67               5/7
e5-gele seul (germe 1)        63/120       56       52               4/7
setfit seul (germe 1)         74/120       49       43               6/7

exactitude du nouveau témoin, tous germes (accord exact sur les 120) :
  tfidf     [49, 58, 53, 54, 45]  → min 45, max 58
  e5-gele   [63, 67, 63, 65, 59]  → min 59, max 67
  setfit    [74, 69, 69, 71, 66]  → min 66, max 74


## L'indépendance des erreurs — le cœur du sujet

**Un témoin qui se trompe là où les autres se trompent n'alarme jamais.** C'est ce qui départage les
montages, davantage que l'exactitude : le lexique est un moteur de surface lexicale, TF-IDF est le
même matériau, et le plongement dense est censé être d'une autre nature.

Le résultat le plus dérangeant du dossier, dû à Abney et relevé par
[Protocole de mesure](https://github.com/AmauryTISSOT/microservice_rgpd/issues/45) : **le témoignage
par désaccord est le plus fiable quand on en a le moins besoin** — la valeur d'un témoin dépend de
la qualité du moteur qu'il contrôle. Avec 7 erreurs sur 120, `qwen3:8b` est très bon, et c'est
précisément ce qui rend le témoignage difficile à valoriser.

In [17]:
def indicateur_erreur(avis):
    return np.array([0 if accord(avis[i], verite[i]) else 1 for i in range(N_EXEMPLES)])


def phi(a, b):
    """Corrélation de Matthews entre deux vecteurs d'erreur. 0 = erreurs indépendantes ;
    proche de 1 = les deux moteurs se trompent sur les mêmes exemples."""
    n11 = int(((a == 1) & (b == 1)).sum()); n10 = int(((a == 1) & (b == 0)).sum())
    n01 = int(((a == 0) & (b == 1)).sum()); n00 = int(((a == 0) & (b == 0)).sum())
    denominateur = math.sqrt((n11 + n10) * (n01 + n00) * (n11 + n01) * (n10 + n00))
    return (n11 * n00 - n10 * n01) / denominateur if denominateur else float("nan"), (n11, n10, n01, n00)


err_V, err_L = indicateur_erreur(V), indicateur_erreur(L)
print(f"{'couple':<28} {'φ':>7}   erreurs communes / seul A / seul B / aucun")
valeur, table = phi(err_V, err_L)
print(f"{'qwen3:8b × lexique':<28} {valeur:>7.3f}   {table}")

for montage in ENREGISTREMENTS:
    par_id = {r["id"]: set(r["predit"]) for r in ENREGISTREMENTS[montage]
              if r["germe"] == GERMES[0]}
    err_N = indicateur_erreur([par_id[i] for i in identifiants])
    v_llm, t_llm = phi(err_V, err_N)
    v_lex, t_lex = phi(err_L, err_N)
    print(f"{'qwen3:8b × ' + montage:<28} {v_llm:>7.3f}   {t_llm}")
    print(f"{'lexique × ' + montage:<28} {v_lex:>7.3f}   {t_lex}")

print()
print("Sur les 7 erreurs de qwen3:8b, combien le nouveau témoin commet-il aussi ?")
for montage in ENREGISTREMENTS:
    compte = []
    for germe in GERMES:
        par_id = {r["id"]: set(r["predit"]) for r in ENREGISTREMENTS[montage] if r["germe"] == germe}
        compte.append(sum(1 for i in erreurs_llm if not accord(par_id[identifiants[i]], verite[i])))
    print(f"  {montage:9s} {compte} sur 7 — plus c'est bas, plus le témoin est utile")
print(f"  {'lexique':9s} {sum(1 for i in erreurs_llm if not accord(L[i], verite[i]))} sur 7")

couple                             φ   erreurs communes / seul A / seul B / aucun
qwen3:8b × lexique             0.042   (2, 5, 24, 89)
qwen3:8b × tfidf              -0.010   (4, 3, 67, 46)
lexique × tfidf                0.231   (21, 5, 50, 44)
qwen3:8b × e5-gele             0.119   (5, 2, 52, 61)
lexique × e5-gele              0.148   (16, 10, 41, 53)
qwen3:8b × setfit              0.023   (3, 4, 43, 70)
lexique × setfit               0.168   (14, 12, 32, 62)

Sur les 7 erreurs de qwen3:8b, combien le nouveau témoin commet-il aussi ?
  tfidf     [4, 4, 3, 4, 3] sur 7 — plus c'est bas, plus le témoin est utile
  e5-gele   [5, 5, 4, 5, 4] sur 7 — plus c'est bas, plus le témoin est utile
  setfit    [3, 2, 4, 2, 5] sur 7 — plus c'est bas, plus le témoin est utile
  lexique   2 sur 7


## L'invariant I2 avant contrainte — le témoin a-t-il *saisi* l'exclusivité ?

Le taux de violation **après** contrainte est nul par construction et ne départage rien. Le
renseignement est ailleurs : combien de fois le témoin **aurait** violé l'exclusivité si on l'avait
laissé faire. Un témoin qui aurait violé quarante fois est un témoin dont on se méfiera, même
contraint — la contrainte lui serait imposée de l'extérieur plutôt que comprise.

`qwen3:8b` ne viole jamais I2 sur les 120 : il n'y a pas de problème de ligne de base.

**Cela se lit ; cela ne se soustrait pas d'un score.**

In [18]:
print(f"{'montage':<9} {'violations brutes par germe':<28} {'aucun seuil franchi':<22}")
for montage, enregistrements in ENREGISTREMENTS.items():
    violations, aveugles = [], []
    for germe in GERMES:
        du_germe = [r for r in enregistrements if r["germe"] == germe]
        violations.append(sum(1 for r in du_germe if r["violation_i2_brute"]))
        aveugles.append(sum(1 for r in du_germe if r["aucun_seuil_franchi"]))
    print(f"{montage:<9} {str(violations):<28} {str(aveugles):<22}")

print()
print("« aucun seuil franchi » = le témoin ne reconnaît rien et retombe sur le résidu.")
print("C'est un doute, et la règle d'arbitrage le fait ressortir en hors-perimetre :")
print("le seul cas où la sortie a l'apparence d'un verdict alors que c'est un aveu d'ignorance.")

montage   violations brutes par germe  aucun seuil franchi   
tfidf     [0, 0, 0, 0, 0]              [52, 50, 53, 55, 55]  
e5-gele   [11, 4, 8, 7, 8]             [13, 18, 13, 15, 16]  
setfit    [3, 3, 6, 1, 5]              [4, 2, 11, 19, 9]     

« aucun seuil franchi » = le témoin ne reconnaît rien et retombe sur le résidu.
C'est un doute, et la règle d'arbitrage le fait ressortir en hors-perimetre :
le seul cas où la sortie a l'apparence d'un verdict alors que c'est un aveu d'ignorance.


## Les huit paires minimales — diagnostic qualitatif, **jamais** en pourcentage

Ce sont les exemples les plus discriminants du corpus : deux textes quasi identiques dont la
qualification diffère. Le protocole les traite comme des **groupes** — jamais séparés entre
apprentissage et test —, ce qui a été vérifié plus haut.

Un pourcentage sur huit exemples ne veut rien dire — l'intervalle de Wilson sur « 6 sur 8 » est
calculé plus bas, et il couvre la moitié de l'échelle. On lit donc les paires **une par une**, et
rien d'autre.

In [19]:
index = {ident: i for i, ident in enumerate(identifiants)}
for montage in ENREGISTREMENTS:
    print(f"\n{montage}")
    for gauche, droite in PAIRES_MINIMALES:
        ligne = []
        for germe in GERMES:
            par_id = {r["id"]: set(r["predit"]) for r in ENREGISTREMENTS[montage]
                      if r["germe"] == germe}
            bon_g = par_id[gauche] == verite[index[gauche]]
            bon_d = par_id[droite] == verite[index[droite]]
            distingue = par_id[gauche] != par_id[droite]
            ligne.append(("··" if not distingue else ("OO" if bon_g and bon_d
                          else ("O·" if bon_g else ("·O" if bon_d else "xx")))))
        print(f"  {gauche:>7} / {droite:<7} {' '.join(ligne)}   "
              f"vérité {sorted(verite[index[gauche]])} / {sorted(verite[index[droite]])}")
print()
print("OO les deux justes · O· ou ·O un seul juste · xx tous deux faux mais distingués"
      " · ·· les deux membres reçoivent la même réponse (la paire n'est pas distinguée)")


tfidf
   edg-13 / edg-14  ·O O· ·· OO O·   vérité ['rectification'] / ['effacement']
   edg-04 / edg-05  xx xx ·· xx ··   vérité ['hors-perimetre'] / ['effacement']
   eff-02 / hop-03  ·· OO OO OO OO   vérité ['effacement'] / ['hors-perimetre']
   eff-03 / hop-24  ·· ·· ·· xx ··   vérité ['effacement'] / ['hors-perimetre']
   opp-02 / mul-01  xx O· ·· xx xx   vérité ['opposition'] / ['effacement', 'opposition']
   edg-11 / edg-12  OO ·O OO OO ·O   vérité ['opposition'] / ['limitation']
   acc-01 / hop-22  OO ·· OO OO ··   vérité ['acces'] / ['hors-perimetre']
   eff-05 / opp-01  ·· ·· O· ·O ··   vérité ['effacement'] / ['opposition']

e5-gele
   edg-13 / edg-14  ·· ·O ·O ·O ·O   vérité ['rectification'] / ['effacement']
   edg-04 / edg-05  ·· ·· OO ·· OO   vérité ['hors-perimetre'] / ['effacement']
   eff-02 / hop-03  OO OO ·O OO OO   vérité ['effacement'] / ['hors-perimetre']
   eff-03 / hop-24  ·· OO xx O· ··   vérité ['effacement'] / ['hors-perimetre']
   opp-02 / mul-01  O· O· O· 

## La calibration — ce que le montage sait dire de son doute

[Le ticket #46](https://github.com/AmauryTISSOT/microservice_rgpd/issues/46) a rendu cette question
centrale : la `DeclaredConfidence` de `qwen3:8b` est **dégénérée** — 118 `Low`, 2 `High`, aucun
`Medium` sur 120. L'échelle ordinale à trois degrés ne discrimine rien en pratique. Un témoin
incapable de mieux dire son doute reproduirait la limite qu'il est censé lever.

**`predict_proba` brut, aucun recalibrage.** Le recalibrage a posteriori est fermé par le volume :
la doc `scikit-learn` disqualifie la régression isotone sous ~1000 exemples, le sigmoïde de Platt
suppose une erreur de calibration symétrique que notre déséquilibre interdit, et
`CalibratedClassifierCV(ensemble=True)` casse mécaniquement avec 13 exemples pour `portabilite`.
**Mesurer plutôt que corriger** est la seule option honnête : un recalibrage ajusté sur un jeu
minuscule produirait une confiance *qui a l'air* calibrée.

Les trois montages partagent la même tête : la comparaison de calibration est donc à tête
constante — une mesure que la littérature n'a pas publiée pour SetFit, dont #43 supposait seulement
la calibration dégradée par l'objectif contrastif.

In [20]:
def brier_et_logloss(enregistrements, montage, germe):
    par_id = {r["id"]: r["probabilites"] for r in enregistrements
              if r["montage"] == montage and r["germe"] == germe}
    briers, pertes = [], []
    for l, slug in enumerate(SLUGS):
        p = np.array([par_id[i][slug] for i in identifiants])
        y = Y[:, l]
        briers.append(float(np.mean((p - y) ** 2)))
        pc = np.clip(p, 1e-12, 1 - 1e-12)
        pertes.append(float(-np.mean(y * np.log(pc) + (1 - y) * np.log(1 - pc))))
    return float(np.mean(briers)), float(np.mean(pertes)), briers


print(f"{'montage':<9} {'Brier moyen':>12} {'perte log moyenne':>19}   (moyennes non pondérées "
      f"sur les 7 têtes, hors-pli)")
for montage, enregistrements in ENREGISTREMENTS.items():
    b = [brier_et_logloss(enregistrements, montage, g)[0] for g in GERMES]
    p = [brier_et_logloss(enregistrements, montage, g)[1] for g in GERMES]
    print(f"{montage:<9} {np.mean(b):>12.4f} {np.mean(p):>19.4f}")

print(f"\nBrier par étiquette (germe {GERMES[0]}) — plus bas est meilleur :")
print(f"{'montage':<9} " + " ".join(f"{s[:6]:>7}" for s in SLUGS))
for montage, enregistrements in ENREGISTREMENTS.items():
    _, _, briers = brier_et_logloss(enregistrements, montage, GERMES[0])
    print(f"{montage:<9} " + " ".join(f"{b:>7.3f}" for b in briers))

print(f"\nMarge (écart entre la plus haute probabilité et la suivante), médiane par montage :")
for montage, enregistrements in ENREGISTREMENTS.items():
    marges = [r["marge"] for r in enregistrements if r["germe"] == GERMES[0]]
    print(f"  {montage:<9} médiane {np.median(marges):.3f}  "
          f"quartiles [{np.percentile(marges, 25):.3f} – {np.percentile(marges, 75):.3f}]")

montage    Brier moyen   perte log moyenne   (moyennes non pondérées sur les 7 têtes, hors-pli)
tfidf           0.1769              0.5307
e5-gele         0.1285              0.4268
setfit          0.0699              0.2357

Brier par étiquette (germe 20180525) — plus bas est meilleur :
montage     acces  rectif  efface  limita  portab  opposi  hors-p
tfidf       0.177   0.149   0.201   0.149   0.154   0.192   0.205
e5-gele     0.137   0.086   0.127   0.088   0.089   0.127   0.145
setfit      0.092   0.044   0.059   0.042   0.040   0.079   0.101

Marge (écart entre la plus haute probabilité et la suivante), médiane par montage :


  tfidf     médiane 0.011  quartiles [0.000 – 0.081]
  e5-gele   médiane 0.143  quartiles [0.057 – 0.259]
  setfit    médiane 0.474  quartiles [0.269 – 0.723]


## Le coût — mesuré et rapporté, **jamais** éliminatoire

Décision assumée par #48, conforme au § 1 de la spec : *« la latence n'est pas un critère de
rejet »*. Aucun de ces chiffres ne change le oui/non.

La latence est rendue en **médiane et p95**, jamais en moyenne — une moyenne écrase la queue, et
c'est la queue qui affame le lexique. Le danger nommé par le § 5.8 n'est pas la lenteur du service :
c'est le **débranchement du lexique**, donc la mort de l'alarme qu'on cherche justement à
améliorer.

La mesure est faite **texte par texte**, en lot de 1 : c'est le régime du service, pas celui du
banc d'essai.

**Et sur CPU**, alors que le montage 3 s'entraîne sur la carte. Ce n'est pas une incohérence, c'est
la même règle appliquée deux fois : on mesure le coût **là où il serait payé**. Le sidecar déployé
n'a pas de carte à lui — l'ADR-0001 la donne à `qwen3:8b` —, donc rapporter une latence GPU
décrirait un service qui n'existe pas. La durée d'entraînement, elle, est un coût de **banc** : on
la paie une fois, hors production, et rien n'oblige à la payer sur le processeur.

In [21]:
# Un appel à blanc : la première inférence paie l'allocation des tampons.
encodeur.encode([PREFIXE_E5 + textes[0]], normalize_embeddings=True, batch_size=1)

latences_ms = []
for t in textes:
    debut = time.perf_counter()
    encodeur.encode([PREFIXE_E5 + t], normalize_embeddings=True, batch_size=1)
    latences_ms.append((time.perf_counter() - debut) * 1000)

tete_finale = ajuster_tete(PLONGEMENTS_GELES, Y, {"C": 1.0, "class_weight": "balanced"})
debut = time.perf_counter()
for i in range(N_EXEMPLES):
    tete_finale.predict_proba(PLONGEMENTS_GELES[i:i + 1])
latence_tete_ms = (time.perf_counter() - debut) / N_EXEMPLES * 1000

vectoriseur = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), sublinear_tf=True)
vectoriseur.fit(textes)
debut = time.perf_counter()
for t in textes:
    vectoriseur.transform([t])
latence_tfidf_ms = (time.perf_counter() - debut) / N_EXEMPLES * 1000

# La taille des poids se lit sur le modèle **déjà chargé** et sur le cache **déjà rempli** :
# redemander une instantané au Hub relancerait un téléchargement, ce qui n'a rien à voir avec la
# question posée — et échoue sous Windows sans privilège de lien symbolique.
from huggingface_hub import constants

dossier = (Path(constants.HF_HUB_CACHE) / f"models--{ENCODEUR.replace('/', '--')}"
           / "snapshots" / REVISION)
poids = (sum(f.stat().st_size for f in dossier.rglob("*") if f.is_file())
         if dossier.is_dir() else float("nan"))
parametres = sum(p.numel() for p in encodeur.parameters())
octets_parametres = sum(p.numel() * p.element_size() for p in encodeur.parameters())

mediane, p95 = float(np.median(latences_ms)), float(np.percentile(latences_ms, 95))
print(f"encodage e5-small, lot de 1   : médiane {mediane:.1f} ms   p95 {p95:.1f} ms")
print(f"                                (moyenne {np.mean(latences_ms):.1f} ms — écrase la queue, "
      f"ne pas s'en servir)")
print(f"tête logistique, un texte     : {latence_tete_ms:.3f} ms")
print(f"vectorisation TF-IDF, un texte: {latence_tfidf_ms:.3f} ms")
print(f"chargement du modèle          : {SECONDES_CHARGEMENT:.1f} s")
print(f"empreinte mémoire résidente   : {RSS_MODELE / 1024 ** 2:.0f} Mio à la charge du modèle")
print(f"processus complet, à cet instant: {psutil.Process().memory_info().rss / 1024 ** 3:.2f} Gio")
print(f"poids sur disque (hors dépôt)  : {poids / 1024 ** 2:.0f} Mio dans le cache Hugging Face")
print(f"paramètres                    : {parametres / 1e6:.0f} M "
      f"= {octets_parametres / 1024 ** 2:.0f} Mio en mémoire")
print()
print(f"LECTURE EN CONTENTION (§ 5.8 de la spec) — pendant {p95 + latence_tete_ms:.0f} ms, "
      f"le point d'entrée")
print(f"du lexique et son échéance de 5 s sont exposés. Le sidecar sert les deux moteurs dans le")
print(f"même processus ; toute famille sauf le sac de mots est liée au CPU, et le coût n'est donc")
print(f"pas seulement une latence ajoutée : c'est une contention avec un moteur existant.")

encodage e5-small, lot de 1   : médiane 34.0 ms   p95 41.9 ms
                                (moyenne 35.8 ms — écrase la queue, ne pas s'en servir)
tête logistique, un texte     : 1.821 ms
vectorisation TF-IDF, un texte: 0.763 ms
chargement du modèle          : 4.2 s
empreinte mémoire résidente   : 267 Mio à la charge du modèle
processus complet, à cet instant: 2.08 Gio
poids sur disque (hors dépôt)  : 470 Mio dans le cache Hugging Face
paramètres                    : 118 M = 449 Mio en mémoire

LECTURE EN CONTENTION (§ 5.8 de la spec) — pendant 44 ms, le point d'entrée
du lexique et son échéance de 5 s sont exposés. Le sidecar sert les deux moteurs dans le
même processus ; toute famille sauf le sac de mots est liée au CPU, et le coût n'est donc
pas seulement une latence ajoutée : c'est une contention avec un moteur existant.


### Le journal des dépendances ajoutées au sidecar

Ce que coûterait, en dépendances, le passage du montage 2 ou 3 en production. Le sidecar servi
n'en porte aujourd'hui que quatre — `fastapi`, `openai`, `pydantic`, `uvicorn`.

**Ollama n'apparaît pas dans ce journal, et c'est une décision.** L'option « zéro dépendance » —
réutiliser le client `openai` déjà présent pour taper `/v1/embeddings` — ne coûte aucun paquet,
mais `/v1/embeddings` ne transmet pas `options` (donc pas de `num_gpu: 0`) et certaines variables
Ollama sont globales. Le troisième moteur entrerait alors en **couplage avec le moteur de
verdict**, ce que l'ADR-0001 refuse explicitement : deux points d'entrée séparés existent
précisément pour que l'indépendance des avis soit structurelle et non déclarative. *Un témoin qui
partage son processus d'inférence avec celui qu'il contrôle n'est plus tout à fait un témoin.*

In [22]:
from importlib.metadata import distributions

interessants = {"torch", "transformers", "sentence-transformers", "scikit-learn", "setfit",
                "tokenizers", "safetensors", "huggingface-hub", "numpy", "scipy", "datasets"}
tailles = {}
for distribution in distributions():
    nom = (distribution.metadata["Name"] or "").lower()
    if nom in interessants:
        tailles[nom] = distribution.version

print("Dépendances directes qu'un montage dense ajouterait au sidecar :")
for nom in sorted(tailles):
    print(f"  {nom:<22} {tailles[nom]}")
venv = Path(sys.prefix)
octets = sum(f.stat().st_size for f in (venv / "Lib" / "site-packages").rglob("*")
             if f.is_file()) if (venv / "Lib" / "site-packages").is_dir() else 0
print(f"\nEnvironnement d'exploration complet : {octets / 1024 ** 3:.2f} Gio sur disque")
print("Le sidecar servi en porte quatre : fastapi, openai, pydantic, uvicorn.")

Dépendances directes qu'un montage dense ajouterait au sidecar :
  datasets               5.0.1
  numpy                  2.5.1
  safetensors            0.8.0
  scikit-learn           1.9.0
  scipy                  1.18.0
  sentence-transformers  5.6.1
  setfit                 1.1.3
  tokenizers             0.22.2
  torch                  2.13.0+cu126
  transformers           4.57.6



Environnement d'exploration complet : 4.49 Gio sur disque
Le sidecar servi en porte quatre : fastapi, openai, pydantic, uvicorn.


## Ce que 120 exemples ne permettent pas de conclure

À dire plutôt qu'à laisser deviner.

**L'ordre de grandeur, établi par [Protocole de mesure](https://github.com/AmauryTISSOT/microservice_rgpd/issues/45)
avant tout chiffre : ±7 points sur une métrique globale, ±20 à 25 points sur une étiquette à 13-14
exemples.** Tout écart par classe rare inférieur à 20 points est du **bruit**.

Les trois étiquettes rares — `portabilite` (13), `rectification` (14), `limitation` (14) — laissent
2 à 3 exemples par pli de test. Un écart d'un seul exemple s'y lit comme 30 à 50 points de rappel.
**Rien de ce qui s'y joue ne porte une décision.**

Le **rappel d'alarme ne peut rien décider** : 7 erreurs, dont 5 déjà attrapées, laissent un gisement
de 2 exemples. C'est pourquoi le critère porte uniquement sur les fausses alarmes.

Les **huit paires minimales** se lisent une par une, jamais en pourcentage.

Enfin, le corpus est celui sur lequel le **lexique a été écrit**. Sa performance y est donc lue avec
la bienveillance d'un jeu qu'il connaît, là où le nouveau témoin est mesuré **hors-pli**. La
comparaison *nouveau seul contre lexique seul* est asymétrique en faveur du lexique, et c'est une
raison de plus de ne pas la laisser peser sur le verdict.

In [23]:
print("Étiquettes rares : exemples par pli de test, par germe")
for l, slug in enumerate(SLUGS):
    if Y[:, l].sum() > 20:
        continue
    par_germe = [[int(Y[PLIS[g] == f].sum(axis=0)[l]) for f in range(K_EXTERNE)] for g in GERMES]
    print(f"  {slug:<15} total {Y[:, l].sum():>3} → {par_germe[0]} … "
          f"minimum sur tous les germes : {min(min(p) for p in par_germe)}")

bas, haut = wilson(6, 8)
print(f"\nIntervalle de Wilson sur « 6 paires minimales sur 8 » : "
      f"[{bas:.2f} – {haut:.2f}] — d'où la lecture une par une.")
bas, haut = wilson(1, 2)
print(f"Intervalle de Wilson sur « 1 erreur rattrapée sur 2 »  : "
      f"[{bas:.2f} – {haut:.2f}] — d'où l'abandon du rappel d'alarme comme axe.")

Étiquettes rares : exemples par pli de test, par germe
  rectification   total  14 → [2, 3, 3, 3, 3] … minimum sur tous les germes : 2
  limitation      total  14 → [3, 2, 3, 3, 3] … minimum sur tous les germes : 2
  portabilite     total  13 → [3, 3, 3, 2, 2] … minimum sur tous les germes : 2
  opposition      total  20 → [4, 4, 4, 4, 4] … minimum sur tous les germes : 3

Intervalle de Wilson sur « 6 paires minimales sur 8 » : [0.41 – 0.93] — d'où la lecture une par une.
Intervalle de Wilson sur « 1 erreur rattrapée sur 2 »  : [0.09 – 0.91] — d'où l'abandon du rappel d'alarme comme axe.


## La conclusion, confrontée au critère

Le critère est appliqué **mécaniquement**, tel qu'il a été écrit avant que le moindre chiffre
n'existe. Aucune latitude n'est prise ici : un intervalle qui chevauche 21,2 % est un **non**, pas
une zone grise, et « non concluant » est réservé à l'échec d'une condition de validité de la
**liste close**.

Le candidat principal de la carte est le **montage 2** (`e5-small` gelé). Le montage 1 est une
ligne de base et le montage 3 un échelon supérieur : ils éclairent la lecture, ils ne sont pas
l'objet du critère.

La lecture de ces chiffres et le verdict appartiennent à
[Lire les chiffres du notebook et prononcer le verdict](https://github.com/AmauryTISSOT/microservice_rgpd/issues/52).

In [24]:
CONDITIONS_DE_VALIDITE = {
    "bug d'accents confirmé sur le modèle retenu": BUG_ACCENTS,
    "validation croisée impraticable (plis, paires minimales, réglage)": not DECOUPAGE_VALIDE,
    "prédictions constantes ou dégénérées": not PREDICTIONS_NON_DEGENEREES,
}
echecs = [cause for cause, survenue in CONDITIONS_DE_VALIDITE.items() if survenue]

print("Conditions de validité (liste close) :")
for cause, survenue in CONDITIONS_DE_VALIDITE.items():
    print(f"  [{'ÉCHEC' if survenue else '  ok '}] {cause}")

for montage in ("e5-gele", "setfit", "tfidf"):
    par_germe = CRITERE[montage]
    pire = max(par_germe.values(), key=lambda r: r["fausses_alarmes"])
    pire_c2 = all(r["condition_2"] for r in par_germe.values())
    print(f"\n───────── {montage}"
          f"{'   ← le candidat principal' if montage == 'e5-gele' else ''}")
    print(f"  au pire germe : {pire['fausses_alarmes']} fausses alarmes sur {len(corrects_llm)}"
          f" = {pire['taux']:.1%}, Wilson [{pire['wilson'][0]:.1%} – {pire['wilson'][1]:.1%}]")
    print(f"  condition 1 — borne haute {pire['wilson'][1]:.1%} < {TAUX_A:.1%} ? "
          f"{'OUI' if pire['condition_1'] else 'NON'}")
    manquantes = set()
    for r in par_germe.values():
        manquantes |= ERREURS_ATTRAPEES_PAR_A - r["erreurs_conservees"]
    print(f"  condition 2 — les 5 erreurs conservées à tous les germes ? "
          f"{'OUI' if pire_c2 else 'NON, perdues : ' + str(sorted(manquantes))}")
    print(f"  condition 3 — appliquée : les deux ci-dessus sont lues au pire des "
          f"{len(GERMES)} germes")

    if echecs:
        verdict = "NON CONCLUANT — " + " ; ".join(echecs)
    elif pire["condition_1"] and pire_c2:
        verdict = "OUI"
    else:
        verdict = "NON"
    print(f"  ⇒ ce que le critère commande : {verdict}")

print(f"\n(Le montage 1 est une ligne de base et le montage 3 un échelon supérieur : le critère de"
      f"\n la carte porte sur le candidat principal, e5-gele.)")

Conditions de validité (liste close) :
  [  ok ] bug d'accents confirmé sur le modèle retenu
  [  ok ] validation croisée impraticable (plis, paires minimales, réglage)
  [  ok ] prédictions constantes ou dégénérées

───────── e5-gele   ← le candidat principal
  au pire germe : 17 fausses alarmes sur 113 = 15.0%, Wilson [9.6% – 22.8%]
  condition 1 — borne haute 22.8% < 21.2% ? NON
  condition 2 — les 5 erreurs conservées à tous les germes ? NON, perdues : ['acc-10', 'hop-22']
  condition 3 — appliquée : les deux ci-dessus sont lues au pire des 5 germes
  ⇒ ce que le critère commande : NON

───────── setfit
  au pire germe : 16 fausses alarmes sur 113 = 14.2%, Wilson [8.9% – 21.8%]
  condition 1 — borne haute 21.8% < 21.2% ? NON
  condition 2 — les 5 erreurs conservées à tous les germes ? NON, perdues : ['acc-10', 'hop-22', 'por-08']
  condition 3 — appliquée : les deux ci-dessus sont lues au pire des 5 germes
  ⇒ ce que le critère commande : NON

───────── tfidf
  au pire germe : 22 f

## Écarts au protocole, et ce qui a été découvert en chemin

**Écart 1 — le corps contrastif de SetFit n'est pas réentraîné dans les plis internes.** Déclaré
plus haut, avec son motif chiffré : 125 entraînements au lieu de 25, de l'ordre de 2 h 45 sur cette
carte — et de 27 heures sur ce CPU, chiffre qui était le motif d'origine. Le pli de test externe reste intact, donc les prédictions hors-pli — les seules dont le
critère se nourrisse — le sont aussi.

**Écart 2 — aucun réglage des hyperparamètres de SetFit.** `num_iterations`, la taille de lot et le
nombre d'époques sont ceux publiés par défaut. Les régler aurait ajouté une dimension à une
recherche déjà imbriquée, sur 120 exemples ; un montage 3 décevant est donc un montage 3 **non
réglé**, ce que la brume « une seconde itération » de la carte a précisément prévu.

**Écart 3 — la stratification itérative retenue est du premier ordre.** #45 laissait l'arbitrage
avec la variante de second ordre « mesurable, pas doctrinal », et il ne porte que sur les 19
exemples multi-étiquettes. Il n'a pas été mesuré : c'est un travail à part entière pour un effet
borné par construction à 19 exemples sur 120.

**Écart 4 — le corps contrastif est entraîné sur GPU, le reste sur CPU.** Les 25 entraînements
saturaient les huit cœurs physiques pendant près de six heures d'affilée, jusqu'à rendre la machine
inutilisable ; la bascule sur la carte les ramène à une demi-heure. Ce que cet écart
**n'atteint pas** : la latence rapportée reste mesurée sur CPU, parce que c'est le coût du service
et que le service n'a pas de carte à lui. Ce qu'il **coûte** : deux exécutions sur des appareils
différents ne rendront pas les mêmes décimales au montage 3 — l'exigence annoncée à ce niveau était
déjà le *verdict*, pas les décimales, et la cellule d'environnement imprime l'appareil pour que
l'écart reste diagnosticable.

**Aucun écart sur ce qui compte.** Le réglage de la tête est imbriqué, les prédictions sont
hors-pli, I2 est appliqué par construction dans le modèle, les paires minimales ne sont jamais
séparées, R = 5 germes sont lus au pire, et le critère est celui écrit avant les chiffres.

### Ce que ce notebook a découvert et que personne ne savait

- La **durée réelle** d'un entraînement contrastif SetFit, sur CPU **et** sur GPU, et donc des 25 —
  la brume que la carte avait explicitement laissée à ce ticket. Le rapport entre les deux, mesuré
  et non estimé, est ce qui rend l'échelon SetFit praticable ou non hors d'une machine de calcul.
- Une comparaison de **calibration à tête constante** entre plongement gelé et plongement
  contrastif, que la littérature n'a pas publiée.
- Le **taux de violation brute d'I2** : si le montage a saisi l'exclusivité de `hors-perimetre`, ou
  si la contrainte lui est imposée de l'extérieur.
- Combien de fois le témoin **ne reconnaît rien** et retombe sur le résidu — le doute déguisé en
  verdict, que #47 redoutait.